# 10 — Fetch pinned COSMoS sources

Pins one upstream commit of `cdisc-org/COSMoS` and downloads the four inputs this
repo depends on into `../downloads/`, each with a provenance sidecar.

`cdisc-org/COSMoS` publishes no tags and no releases, so the pin is a **commit
SHA**. One SHA co-pins all four inputs; bumping it is a deliberate action — see
`CLAUDE.md`.

Never fetch from `cdisc-org.github.io/COSMoS/export/…`: that is the current
publication and is not pinnable. Never fetch from the CDISC Library API: it is
member-gated.

## Pinned configuration

In [ ]:
COSMOS_REPO        = "cdisc-org/COSMoS"
COSMOS_COMMIT      = "031429b1d14823721991cd23ee88a11616686ce3"
COSMOS_COMMIT_DATE = "2026-07-21T14:35:11Z"

# Part of the pin, recorded from the upstream commit and verified in
# docs/source-verification.md. Not derived at build time — the fetch must not
# depend on api.github.com.

INPUTS = {
    "bc_export":  "export/cdisc_biomedical_concepts_latest.csv",
    "dss_export": "export/cdisc_sdtm_dataset_specializations_latest.csv",
    "bc_model":   "model/cosmos_bc_model.yaml",
    "sdtm_model": "model/cosmos_sdtm_model.yaml",
}

SIDECARS = {
    "bc_export":  ".fetch_meta_bc_export.json",
    "dss_export": ".fetch_meta_dss_export.json",
    "bc_model":   ".fetch_meta_bc_model.json",
    "sdtm_model": ".fetch_meta_sdtm_model.json",
}

DOWNLOADS = "../downloads"

## Build the raw URLs

In [ ]:
RAW_BASE = f"https://raw.githubusercontent.com/{COSMOS_REPO}/{COSMOS_COMMIT}"
URLS = {key: f"{RAW_BASE}/{path}" for key, path in INPUTS.items()}

for key, url in URLS.items():
    print(f"{key:11s} {url}")

## Fetch via `urllib` (standard library)

In [ ]:
import urllib.request
from pathlib import Path

Path(DOWNLOADS).mkdir(parents=True, exist_ok=True)

for key, path in INPUTS.items():
    url = URLS[key]
    req = urllib.request.Request(url)
    with urllib.request.urlopen(req) as resp:
        if resp.status != 200:
            raise RuntimeError(f"unexpected status {resp.status} for {url}")
        body = resp.read()
    target = Path(DOWNLOADS) / Path(path).name
    target.write_bytes(body)
    print(f"{key:11s} wrote {len(body):>9,} bytes to {target}")

## Provenance sidecars

One `.fetch_meta_*.json` per input: source URL, pinned commit, SHA-256, size,
retrieval timestamp. For the two CSV exports the **package date** is derived from
the data as `max(package_date)` — `package_date` is a column, not file-level
metadata, and both exports are cumulative.

In [ ]:
import csv
import datetime
import hashlib
import json
from pathlib import Path

fetched_at = datetime.datetime.now(datetime.timezone.utc).isoformat()

for key, path in INPUTS.items():
    target = Path(DOWNLOADS) / Path(path).name
    raw = target.read_bytes()

    meta = {
        "cosmos_repo":        COSMOS_REPO,
        "cosmos_commit":      COSMOS_COMMIT,
        "cosmos_commit_date": COSMOS_COMMIT_DATE,
        "source_path":        path,
        "raw_url":            URLS[key],
        "sha256":             hashlib.sha256(raw).hexdigest(),
        "size_bytes":         len(raw),
        "fetched_at_utc":     fetched_at,
    }

    if key.endswith("_export"):
        rows = list(csv.DictReader(raw.decode("utf-8").splitlines()))
        if not rows:
            raise RuntimeError(f"{target} parsed to zero rows")
        package_dates = sorted({row["package_date"] for row in rows})
        meta["row_count"] = len(rows)
        meta["column_count"] = len(rows[0])
        meta["package_date"] = package_dates[-1]
        meta["package_dates_present"] = package_dates

    sidecar = Path(DOWNLOADS) / SIDECARS[key]
    sidecar.write_text(json.dumps(meta, indent=2) + "\n")
    print(f"{key:11s} sha256 {meta['sha256'][:16]}…  ->  {sidecar.name}")

## Confirm the source shape

Two kinds of check. **Invariants** the downstream pipeline depends on are
fail-fast: if one breaks, the assumption behind P3 has changed and the code that
relies on it must change too. **Counts** vary by package and are printed, not
asserted.

The invariants checked here, with the reasoning in `docs/known-gaps.md` §5:

- rows of one Dataset Specialization are **contiguous**, so the nested DSS node
  can be rebuilt by grouping;
- `subset_codelist` holds **no stringified dict**, so the latent flattening loss
  is still latent;
- the reification quad is never **partially** populated;
- there is **no order column**, so variable order rests on file row order alone
  (decision D5, still open).

In [ ]:
import pandas as pd
from pathlib import Path

bc = pd.read_csv(
    Path(DOWNLOADS) / Path(INPUTS["bc_export"]).name, dtype=str, keep_default_na=False
)
dss = pd.read_csv(
    Path(DOWNLOADS) / Path(INPUTS["dss_export"]).name, dtype=str, keep_default_na=False
)

# Invariants — fail fast.
key = dss["domain"] + "|" + dss["vlm_group_id"]
blocks = (key != key.shift()).cumsum().nunique()
if blocks != key.nunique():
    raise RuntimeError(
        f"DSS rows are interleaved: {blocks} row blocks for {key.nunique()} groups"
    )

stringified = int(dss["subset_codelist"].str.startswith("{").sum())
if stringified:
    raise RuntimeError(f"{stringified} subset_codelist values are stringified dicts")

QUAD = ["subject", "linking_phrase", "predicate_term", "object"]
filled = (dss[QUAD] != "").sum(axis=1)
partial = int(((filled > 0) & (filled < 4)).sum())
if partial:
    raise RuntimeError(f"{partial} rows carry a partial reification quad")

if "v_order" in dss.columns:
    raise RuntimeError("v_order column is present — decision D5 needs revisiting")

print("invariants ok: contiguous groups, no stringified dicts, no partial quads, no v_order")
print()

# Counts — package-dependent, reported not asserted.
fan = dss.groupby("bc_id")["vlm_group_id"].nunique()
print(f"BC export           {len(bc):>6,} rows x {len(bc.columns)} columns")
print(f"DSS export          {len(dss):>6,} rows x {len(dss.columns)} columns")
print(f"package date        {max(bc['package_date'].max(), dss['package_date'].max())}")
print(f"distinct bc_id      {bc['bc_id'].nunique():>6,}")
print(f"distinct DSS        {dss['vlm_group_id'].nunique():>6,}  in {dss['domain'].nunique()} domains")
print(f"BCs with a DSS      {dss['bc_id'].nunique():>6,}")
print(f"BCs fanning out     {int((fan > 1).sum()):>6,}  max {int(fan.max())}:1 ({fan.idxmax()})")
print(f"complete quads      {int((filled == 4).sum()):>6,}   empty {int((filled == 0).sum()):,}")
print(f"value_list with ';' {int(dss['value_list'].str.contains(';').sum()):>6,}")
print(f"assigned_value ';'  {int(dss['assigned_value'].str.contains(';').sum()):>6,}")

## Provenance

Fetched from `cdisc-org/COSMoS` at the commit pinned in the first code cell.
Sidecars in `../downloads/` carry the URL, SHA-256 and retrieval timestamp per
file; the package date is derived from the exports.

`downloads/` is gitignored. Reproducibility comes from the pin plus the recorded
checksums, not from committing the sources — the same convention as `usdm-rdf`.